<p align="center">
  <img src="foresight.png" alt="FORESIGHT" width="320">
</p>

# Ensemble forecast evaluation

This notebook shows how to use `ForecastPerformance` to evaluate **ensemble forecasts** with a suite of skill metrics.

In [ ]:
import pandas as pd
from pathlib import Path
from performance import ForecastPerformance, Results 

test_dataset_path = Path(r'tests\test_datasets_daily')

obs_path = test_dataset_path / 'obs.parquet'
obs = pd.read_parquet(obs_path)
display(obs.head(5))

ens_path = test_dataset_path / 'ens.parquet'
ens = pd.read_parquet(ens_path)
display(ens.head(5))

In [ ]:
mean = ens.unstack('ensemble_member').mean(axis=1)
median = ens.unstack('ensemble_member').median(axis=1)

results = Results('Model', 'Metric', 'Leadtime')

fp = ForecastPerformance(obs)
fp.add(mean.droplevel('event_datetime').unstack('leadtime'), name='mean')
fp.add(median.droplevel('event_datetime').unstack('leadtime'), name='median')
fp.add(ens, name='ens')

metrics = [fp.KGEprime, fp.NSE, fp.RMSE, fp.MAE, fp.bias, fp.relative_bias, fp.Pearson]

for model in fp.names():
    print(f'{model}')
    for metric in metrics:
        for leadtime in median.index.get_level_values('leadtime').unique():
            results.append(Model=model, Metric=metric.__name__, Leadtime=leadtime,
                        Value=fp.deterministic(metric, model, leadtime=leadtime),
                        )
        
df = results.to_pandas(index=['Metric', 'Model'], columns=['Leadtime'])
display(df)

In [ ]:
results = Results('Model', 'Metric', 'Leadtime')

metrics = ['crps', 'fair_crps', 'reliability', 'resolution']
fp.add(ens + 100, name='ens + 100')
fp.add(ens * 0.95, name='ens * 0.95')

for model in [n for n in fp.names() if n.startswith('ens')]:
    print(f'{model}')
    for metric in metrics:
        for leadtime in ens.index.get_level_values('leadtime').unique():
            results.append(Model=model, Metric=metric, Leadtime=leadtime,
                        Value=fp.probabilistic(metric, model, leadtime=leadtime),
                        )

df = results.to_pandas(index=['Metric', 'Model'], columns=['Leadtime'])
display(df)

In [ ]:
fp.qq_plot(name='ens')